# Hello World of Optimization! 
Let's Build an EV Charging Network
## Our task...
We are in charge of deploying EV charging stations for a municipal parking authority.
- Management would like to install these new charging stations using the available resources from our current budget and infrastructure capacity.
- Right now, we are asked to **maximize the total number of vehicles we can serve per day** with our charging network.
- The charging station types we can install are Level 1 (120V), Level 2 (240V), DC Fast 50kW, DC Fast 150kW, and DC Ultra-Fast 350kW.
- Each type of charging station requires different amounts of several limited resources.
- We are constrained by our available **budget, electrical capacity, physical space, and equipment units**.

## Initial questions
- What are our decisions?
- What are our constraints? 
- What is the objective?
- What data do we need right now?

## The Model
A lot of programming (in Python) is *imperative* -- just providing sequential instructions to complete. But mathematical optimization (aka math programming) is *declarative*. The math programming model does not tell the Gurobi solver what to do specifically. Instead, the model tells the Gurobi solver what the solution must look like. Gurobi then finds the solution in its own way.

So math programming always starts with the creation of a new model. We then add to it the *declarations* about the final solution. 

Math optimization models take two forms:
- The **formulation**: An algebraic representation of the model (what we saw in our introduction!)
- The **code**: Writing the formulation in syntax to some software package.

In [379]:
#%pip install gurobipy
#%pip install plotly
#%pip install nbformat
%pip install gurobi_ml

import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import plotly.express as px



ERROR: Could not find a version that satisfies the requirement gurobi_ml (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for gurobi_ml
Note: you may need to restart the kernel to use updated packages.


In [380]:
model = gp.Model("EV Charging Network")

### The Variables

The model needs something to receive the solution values that the solver finds. It declares *variables* to hold those solution values. The mathematical optimization model never explicitly sets these values. It just describes them and makes rules about the values they can hold. This model declares a variable for each type of charging station.

Let $x_i$ be the number of charging stations installed of type $i \in \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$.

First, let's create data for each charging station type and its characteristics.

In [381]:
chargers = ["Level1", "Level2", "Fast50", "Fast150", "Ultra350"]
data = pd.Series([3, 6, 32, 48, 80], 
                  index=chargers, 
                  name='vehicles_per_day')
data

Level1       3
Level2       6
Fast50      32
Fast150     48
Ultra350    80
Name: vehicles_per_day, dtype: int64

#### Anyone know another way to hardcode this another way?... using a `gurobipy` object?
**Multidict** function splits one dictionary into multiple, allowing for easier access to data while model building

In [382]:
chargers, vehicles_per_day = gp.multidict(
    {
        "Level1":     3,
        "Level2":     6,
        "Fast50":    32,
        "Fast150":   48,
        "Ultra350":  80
    }
)

In [383]:
print(chargers)

['Level1', 'Level2', 'Fast50', 'Fast150', 'Ultra350']


In [384]:
print(vehicles_per_day)

{'Level1': 3, 'Level2': 6, 'Fast50': 32, 'Fast150': 48, 'Ultra350': 80}


Next, variables are added to the model -- one for each charger type. The main types of decision variables are
- CONTINUOUS
- INTEGER
- BINARY

But we'll explore more types later.

#### What type of decision variable should we use in this case?
- Our decisions are the *number of charging stations* we should install for each type

In [385]:
x = model.addVars(chargers, vtype = GRB.INTEGER, name = "chargers")
model.update()
x

{'Level1': <gurobi.Var chargers[Level1]>,
 'Level2': <gurobi.Var chargers[Level2]>,
 'Fast50': <gurobi.Var chargers[Fast50]>,
 'Fast150': <gurobi.Var chargers[Fast150]>,
 'Ultra350': <gurobi.Var chargers[Ultra350]>}

### The Objective Function:

The math model must describe an objective also using algebra. When the model is solved, the value of the objective function will be the maximum or minimum possible while following the rules described in the constraints.

This model will maximize the total number of vehicles served per day. The objective function multiplies the quantity of each charger type installed times its daily vehicle throughput for all charger types.
For example, the total vehicles served by Level 2 chargers is $6*x_{Level2}$.

So the total vehicles served per day is

\begin{equation*}
3*x_{Level1} + 6*x_{Level2} + 32*x_{Fast50} + ... + 80*x_{Ultra350}
\end{equation*}

In [386]:
### Option 1 - Written term by term:
model.setObjective(x["Level1"] * vehicles_per_day["Level1"] + 
                   x["Level2"] * vehicles_per_day["Level2"] + 
                   x["Fast50"] * vehicles_per_day["Fast50"] + 
                   x["Fast150"] * vehicles_per_day["Fast150"] + 
                   x["Ultra350"] * vehicles_per_day["Ultra350"], 
                    sense=GRB.MAXIMIZE)

Let $v_i$ be the vehicles served per day by charger type $i$ and use a bit more math notation:
\begin{equation*}
  \text{Maximize} \space \sum_i v_i*x_i
\end{equation*}

In [387]:
### Option 2 - Here is another way to write the same thing:
model.setObjective(gp.quicksum(x[i] * vehicles_per_day[i] for i in chargers), sense=GRB.MAXIMIZE)

### Option 3 - The prod function is a handy function provided by the Gurobi API.
model.setObjective(x.prod(vehicles_per_day), sense=GRB.MAXIMIZE)

### The Constraints:

This parking authority has limited resources available to deploy the charging stations. The model must make sure to only install chargers for which the resources are available.

Here is a list of resources and how much is available:

In [388]:
resources, available = gp.multidict(
    {
        "budget":     500000,  # dollars
        "power":        2000,  # kW
        "space":         600,  # square meters
        "equipment":     100,  # units
    }
)

In [389]:
print(resources)
print(available)

['budget', 'power', 'space', 'equipment']
{'budget': 500000, 'power': 2000, 'space': 600, 'equipment': 100}


In [390]:
requirements = {
    ("Level1",   "budget"):    500, ("Level1",   "power"):   1.4, ("Level1",   "space"):  12, ("Level1",   "equipment"):  0,
    ("Level2",   "budget"):   3000, ("Level2",   "power"):   7.2, ("Level2",   "space"):  12, ("Level2",   "equipment"):  1,
    ("Fast50",   "budget"):  20000, ("Fast50",   "power"):  50.0, ("Fast50",   "space"):  18, ("Fast50",   "equipment"):  2,
    ("Fast150",  "budget"):  60000, ("Fast150",  "power"): 150.0, ("Fast150",  "space"):  20, ("Fast150",  "equipment"):  4,
    ("Ultra350", "budget"): 120000, ("Ultra350", "power"): 350.0, ("Ultra350", "space"):  25, ("Ultra350", "equipment"):  6,
}

In [391]:
requirements['Level2', 'power']

7.2

Let's introduce **constraints** to make sure we don't use more resources than we have available. Constraints are where the *rules* acting on decision variables are declared.

For example, the amount of power used by all chargers must be less than or equal to the total power capacity available.

\begin{equation*}
\text{total power used} \le 2000
\end{equation*}

Let's write an expression for the total power used using our decision variable, $x_{power}$.
\begin{align*}
\text{total power used} = \space& power_{Level1}* x_{Level1} + power_{Level2}*x_{Level2} + ... + power_{Ultra350}*x_{Ultra350} \\
& power_{Level1}* x_{Level1} + power_{Level2}*x_{Level2} + ... + power_{Ultra350}*x_{Ultra350} \le 2000
\end{align*}

Given we stored this information in the `requirements` dictionary, we let $r_{i,j}$ be the amount of *resource* $j$ required by *charger type* $i$.
\begin{equation*}
  r_{Level1, power} * x_{Level1} + r_{Level2, power}*x_{Level2} + ... + r_{Ultra350, power}*x_{Ultra350} \le 2000
\end{equation*}

In [392]:
### Option 1 - A very explicit way to write these constraints:

# total budget <= budget available
model.addConstr(x["Level1"] * requirements["Level1", "budget"] + x["Level2"] * requirements["Level2", "budget"]
                + x["Fast50"] * requirements["Fast50", "budget"] + x["Fast150"] * requirements["Fast150", "budget"]
                + x["Ultra350"] * requirements["Ultra350", "budget"] <= available["budget"], "budget limit")

# total power <= power available
model.addConstr(x["Level1"] * requirements["Level1", "power"] + x["Level2"] * requirements["Level2", "power"]
                + x["Fast50"] * requirements["Fast50", "power"] + x["Fast150"] * requirements["Fast150", "power"]
                + x["Ultra350"] * requirements["Ultra350", "power"] <= available["power"], "power limit")

# total space <= space available
model.addConstr(x["Level1"] * requirements["Level1", "space"] + x["Level2"] * requirements["Level2", "space"]
                + x["Fast50"] * requirements["Fast50", "space"] + x["Fast150"] * requirements["Fast150", "space"]
                + x["Ultra350"] * requirements["Ultra350", "space"] <= available["space"], "space limit")

# total equipment <= equipment available
model.addConstr(x["Level1"] * requirements["Level1", "equipment"] + x["Level2"] * requirements["Level2", "equipment"]
                + x["Fast50"] * requirements["Fast50", "equipment"] + x["Fast150"] * requirements["Fast150", "equipment"]
                + x["Ultra350"] * requirements["Ultra350", "equipment"] <= available["equipment"], "equipment limit")

<gurobi.Constr *Awaiting Model Update*>

We can also generalize the *quantity* of each resource with $q_j$. Then using a bit more mathematical notation:
\begin{equation*}
  \sum_{i}r_{i, j} * x_{i} \le q_j, \space \text{for all} \space j \space \text{in} \space \{\text{budget, power, space, equipment}\}
\end{equation*}

Note: A short way to write "for all" is using the symbol $\forall$. Also, $\in$ means "in", or "an element of." We saw this notation in the intro.

Where the following can be found using requirements[('Level2', 'power')]
\begin{equation*} r_{Level2, power} \end{equation*}

We can change the code so it looks like this condensed notation and loop through each resource using `quicksum`:

In [393]:
### Option 2 - Using quicksum
for resource in resources:
    model.addConstr(gp.quicksum(x[i] * requirements[i, resource] for i in chargers) <= available[resource], name="resource usage")

A little more compact, and makes it easier to store the constraints as an object:

In [394]:
### Option 3 - Using quicksum and storing constraint as an object
resource_constraints = model.addConstrs((gp.quicksum(x[i] * requirements[i, j] for i in chargers) <= available[j] for j in resources), name="resource usage")

### The Solution:
It's as simple as one line of code to run the optimization, then we query the decision variables for their values (assuming the optimization completed successfully)

In [395]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

WLS license 2660089 - registered to Gurobi Optimization LLC
Optimize a model with 12 rows, 5 columns and 57 nonzeros (Max)
Model fingerprint: 0x230a33dc
Model has 5 linear objective coefficients
Variable types: 0 continuous, 5 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [3e+00, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 5e+05]

Found heuristic solution: objective 150.0000000
Presolve removed 10 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 5 columns, 10 nonzeros
Variable types: 0 continuous, 5 integer (0 binary)

Root relaxation: objective 8.285714e+02, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incum

In [396]:
# use `VarName` and `X` to get t]he variables name and value, respectively:
for v in model.getVars():
    print(f"{v.VarName}: {v.X}")

chargers[Level1]: 9.0
chargers[Level2]: 5.0
chargers[Fast50]: 24.0
chargers[Fast150]: -0.0
chargers[Ultra350]: -0.0


Do you think another (and simpler) approach may work for this specific version of the problem?

## Congratulations!
You have just ran an optimization model!

# More Realistic Problems
Where we left off, a greedy approach would have worked to solve that model. But problems typically have much more complicated business rules to follow, and are larger in scale-- which highlights the strengths of mathematical optimization.

## Before we continue - Let's Create a function to make solving and reporting a bit more straightforward

In [397]:
def solve_and_print_solution(model, x, chargers, sites):
    # Optimize model
    model.setParam("outputflag", 0)
    model.optimize()
    
    # Check the model status
    if model.status == GRB.OPTIMAL:
        print("Optimal solution found\n")
        print('------------------')
        # Print objective value
        print(f"Objective value: {model.objVal}\n")
        print('------------------')
        # Print site-level results
        for s in sites:
            print(f"{s}:")

            # Total chargers at this site
            total_at_site = sum(x[s, c].X for c in chargers)

            if total_at_site == 0:
                print("  No chargers allocated at this site\n")
            else:
                for c in chargers:
                    if x[s, c].X > 0:
                        print(f"  {c}: {x[s, c].X}")
                print()  # blank line for readability

        # --- Metrics (independent of objective) ---

        total_cost = sum(
            chargers_data.installation_cost[j] * x[s, j].X
            for s in sites
            for j in chargers
        )

        total_vehicles = sum(
            chargers_data.vehicles_per_day[j] * x[s, j].X
            for s in sites
            for j in chargers
        )

        total_chargers = sum(
            x[s, j].X
            for s in sites
            for j in chargers
        )

        # --- Print ---
        print(f"Objective Value: {model.objVal:,.0f}")
        print(f"Total Cost: {total_cost:,.0f}")
        print(f"Total Vehicles Served: {total_vehicles:,.0f}")
        print(f"Total Chargers Installed: {total_chargers:,.0f}\n")

    else:
        print(f"Model status: {model.status}")
        #sys.exit(1)

## Let's solve a more realistic business problem 
When setting up the problem the first time we used `multidict` to create the input data for our model. This time, let's read data in from a csv.

If you are on colab, use this cell.

In [398]:
# ### This time, we read in data from csv
# #path = 'https://raw.githubusercontent.com/https://github.com/caroweinberg/ODSC_AI_East_2026/refs/heads/main'
# path = 'https://raw.githubusercontent.com/caroweinberg/ODSC_AI_East_2026/main/'
# available = pd.read_csv(path+'data_files/resource_available.csv', index_col=['resource']).squeeze()
# chargers_data = pd.read_csv(path+'data_files/chargers_data.csv', index_col=['chargers']).squeeze()
# requirements = pd.read_csv(path+'data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()

# resources = available.index.to_list()
# chargers = chargers_data.index.to_list()


If you downloaded the files locally, run this cell:

In [399]:
### This time, we read in data from csv
global_available = pd.read_csv('data_files/global_resources_available.csv', index_col=['resource']).squeeze() # Global resources - Equipment and Budget
site_info = pd.read_csv('data_files/sites.csv', index_col=['site']) # Site resources - Power and Space
chargers_data = pd.read_csv('data_files/chargers_data.csv', index_col=['chargers']).squeeze()
requirements = pd.read_csv('data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()
global_resources = global_available.index.to_list()
site_resources = ['power', 'space']
chargers = chargers_data.index.to_list()
sites = site_info.index.to_list()
site_available_resources = site_info[['power', 'space']].stack().to_dict()

## Create the model 
While the original model was only concerned with one site, a realistic business problem that necessitates mathematical optimization is large in scale. For this use case, that means **multiple candidate sites**. 

- Management would like to install charging stations across these sites using a shared pool of limited resources, including total budget, electrical capacity, physical space, and equipment units.
- Each site has different characteristics and constraints (e.g., varying available space or grid capacity), but all installations draw from the same overall resource pool.
- Our goal is to decide how many chargers of each type to install **at each site**.

For this model: 
- $x_{t,i}$ is the number of charging stations to install of type $i \in {\text{Level1, Level2, Fast50, Fast150, Ultra350}}$ at site $t \in T$.
- $v_i$ is the vehicles served per day by each charger type (assumed the same across all sites).
- $q_j$ is the total amount of resource $j \in {\text{budget, power, space, equipment}}$ available across all sites.
- $q_{t,j}$ is the amount of resource $j$ available at site $t$.
- $r_{i,j}$ is the amount of resource $j$ required to install charger type $i$.
- t is a site in the list of sites, T 

\begin{align*}
\text{Maximize} \quad & \sum_{t \in T} \sum_{i} v_i \cdot x_{t,i} \\

\text{subject to:} \quad & \\

& \sum_{t \in T} \sum_{i} r_{i,j} \cdot x_{t,i} \le q_j, 
\quad \forall j \in \{\text{budget, power, space, equipment}\} \\

& \sum_{i} r_{i,j} \cdot x_{t,i} \le q_{t,j}, 
\quad \forall t \in T,\; j \in \{\text{budget, power, space, equipment}\} \\

& x_{t,i} \ge 0, \quad \forall t \in T,\; \forall i \in \text{Chargers}
\end{align*}

Let's look at where the possible sites are around the Boston Area

In [400]:
# Load data
df = pd.read_csv("data_files/sites.csv")

# Create map
fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    hover_name="site",
    hover_data={"power": True, "space": True},
    zoom=10,
    height=600
)

fig.show()

In [401]:
### Create a new model - this will replace the original 
model = gp.Model("EV Charging Network - Realistic")

### Decision variables
x = model.addVars(sites, chargers, vtype=GRB.INTEGER, name="charger_assignments")

### Global resource constraints (shared across all sites)
global_resource_constraints = model.addConstrs(
    (gp.quicksum(x[t, i] * requirements[i, j] for t in sites for i in chargers)
        <= available[j]
        for j in global_resources
    ),
    name="global_resource_usage"
)

### Site-level constraints for power and physical space 
model.addConstrs(
    (
        gp.quicksum(x[t, i] * requirements[i, j] for i in chargers)
        <= site_available_resources[t, j]
        for t in sites
        for j in site_resources
    ),
    name="site_resource_usage"
)

### Objective function
model.setObjective(
    gp.quicksum(
        x[t, i] * chargers_data.vehicles_per_day[i]
        for t in sites
        for i in chargers
    ),
    sense=GRB.MAXIMIZE
)
### Optimize

solve_and_print_solution(model, x, chargers, sites)

Optimal solution found

------------------
Objective value: 3818.0

------------------
site1:
  Level1: 35.0
  Level2: 6.0

site2:
  Level1: 36.0
  Level2: 1.0

site3:
  Level1: 53.0
  Level2: 1.0

site4:
  Level1: 32.0
  Level2: 1.0

site5:
  Level1: 45.0

site6:
  Level1: 58.0

site7:
  Level1: 28.0
  Level2: 1.0

site8:
  Level1: 62.0

site9:
  Level1: 40.0
  Level2: 1.0

site10:
  Level1: 43.0

site11:
  Level1: 56.0

site12:
  Level1: 35.0

site13:
  Level1: 50.0

site14:
  Level1: 31.0

site15:
  Level1: 64.0
  Level2: 1.0

site16:
  Level1: 48.0

site17:
  Level1: 45.0

site18:
  Level1: 60.0

site19:
  Level1: 37.0
  Level2: 1.0

site20:
  Level1: 64.0

Objective Value: 3,818
Total Cost: 1,922,000
Total Vehicles Served: 3,818
Total Chargers Installed: 935



## Decision expressions
Sometimes it is helpful to store quantities of interest to help make code easier to read, reduce clutter, or get key values quickly.

Suppose we have grouped charger types into two categories: standard and fast. Fast chargers are Fast50, Fast150, and Ultra350, with the rest being standard.

Code the expressions below containing decision variables.

Let's define some sets.
- $I = \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$
- $F = \{\text{Fast50, Fast150, Ultra350}\}$
- $S = I - F$

\begin{align*}
\text{total chargers} &= \sum_{i \in I} x_i, \\
\text{vehicles served by all chargers} &= \sum_{i \in I} v_i*x_i \\
\text{vehicles served by fast chargers} &= \sum_{f \in F} v_f * x_f \\
\text{vehicles served by standard chargers} &= \sum_{s \in S} v_s*x_s \\
\end{align*}

In [402]:
fast_chargers = ['Fast50', 'Fast150', 'Ultra350']
standard_chargers = [i for i in chargers if i not in fast_chargers]

### While we use `quicksum` here, there are more compact ways to write these.

## How many chargers have we placed? 
# Total chargers in all sites
total_chargers = gp.quicksum(x[i,j] for i in sites for j in chargers) # Total chargers in all sites
# Total chargers at each site
total_chargers_per_site = {  # Total chargers at each site
    t: gp.quicksum(x[t, i] for t in sites for i in chargers)
    for t in sites
}

# Fast chargers in all sites
total_fast_chargers = gp.quicksum(x[i,j] for i in sites for j in fast_chargers) # fast chargers in all sites

# Fast chargers at each site
total_fast_chargers_per_site = {  # Total chargers at each site
    t: gp.quicksum(x[t, i] for t in sites for i in fast_chargers)
    for t in sites
}

# Standard chargers in all sites
total_standard_chargers = gp.quicksum(x[i,j] for i in sites for j in standard_chargers) # Standard chargers in all sites
# Standard chargers at each site
standard_chargers_per_site = {
    t: gp.quicksum(x[t, j] for j in standard_chargers)
    for t in sites
}

## How many Vehicles are we serving per day with all the chargers?
# how many vehicles can we serve each day, total?
vehicles_all = gp.quicksum(
    chargers_data.vehicles_per_day[j] * x[t, j]
    for t in sites
    for j in chargers
)

# how many vehicles can we serve each day, at each site?
vehicles_per_site = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[j] * x[t, j]
        for j in chargers
    )
    for t in sites
}

## How many Vehicles are we serving per day with all Fast chargers?
# how many vehicles can we serve each day, total?
vehicles_all_fast = gp.quicksum(
    chargers_data.vehicles_per_day[j] * x[t, j]
    for t in sites
    for j in fast_chargers
)

# how many vehicles can we serve each day, at each site?
vehicles_per_site_fast = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[j] * x[t, j]
        for j in fast_chargers
    )
    for t in sites
}

## How many Vehicles are we serving per day with all Standard chargers?
# how many vehicles can we serve each day, total?
vehicles_all_standard = gp.quicksum(
    chargers_data.vehicles_per_day[j] * x[t, j]
    for t in sites
    for j in standard_chargers
)

# how many vehicles can we serve each day, at each site?
vehicles_per_site_standard = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[j] * x[t, j]
        for j in standard_chargers
    )
    for t in sites
}


# Update model object to apply changes
model.update()

In [403]:
### Test your expressions here
print(vehicles_all_standard.getValue() + vehicles_all_fast.getValue())
print(total_chargers.getValue())

3818.0
935.0


## Changing our model

### New Constraint - Demand
At each site, the chargers installed must serve at least the demand.

$$
\sum_{i \in \text{Chargers}} v_i \cdot x_{t,i} \ge d_s \quad \forall t \in T
$$

In [404]:
demand_fulfillment = model.addConstrs(
    (
        gp.quicksum(
            # Vehicles per day for each charger * # of that type of charger at that site (variable) >= demand 
            chargers_data.vehicles_per_day[i] * x[t, i]
            for i in chargers
        )
        >= site_info['demand'][t]
        for t in sites
    ),
    name="demand_satisfaction"
)

solve_and_print_solution(model, x, chargers,sites)

Optimal solution found

------------------
Objective value: 3818.0

------------------
site1:
  Level1: 35.0
  Level2: 6.0

site2:
  Level1: 36.0
  Level2: 1.0

site3:
  Level1: 53.0
  Level2: 1.0

site4:
  Level1: 32.0
  Level2: 1.0

site5:
  Level1: 45.0

site6:
  Level1: 58.0

site7:
  Level1: 28.0
  Level2: 1.0

site8:
  Level1: 62.0

site9:
  Level1: 40.0
  Level2: 1.0

site10:
  Level1: 43.0

site11:
  Level1: 56.0

site12:
  Level1: 35.0

site13:
  Level1: 50.0

site14:
  Level1: 31.0

site15:
  Level1: 64.0
  Level2: 1.0

site16:
  Level1: 48.0

site17:
  Level1: 45.0

site18:
  Level1: 60.0

site19:
  Level1: 37.0
  Level2: 1.0

site20:
  Level1: 64.0

Objective Value: 3,818
Total Cost: 1,922,000
Total Vehicles Served: 3,818
Total Chargers Installed: 935



### New objective function - Cost
Instead of maximizing the vehicles served, we are asked to minimize the total cost, while respecting demand.

\begin{equation*}
\min \quad \sum_{t \in T} \sum_{i \in I} c_i \cdot x_{t,i}
\end{equation*}

In [405]:
model.setObjective(
    gp.quicksum(
        chargers_data.installation_cost[j] * x[t, j]
        for t in sites
        for j in chargers
    ),
    sense=GRB.MINIMIZE
)
solve_and_print_solution(model, x, chargers,sites)

Optimal solution found

------------------
Objective value: 438000.0

------------------
site1:
  Level1: 3.0

site2:
  Level1: 2.0
  Fast50: 2.0

site3:
  Level1: 8.0

site4:
  Level1: 8.0
  Fast50: 1.0

site5:
  Level1: 2.0

site6:
  Level1: 5.0
  Fast50: 1.0

site7:
  Level1: 3.0
  Fast50: 1.0

site8:
  Fast50: 1.0

site9:
  Level1: 4.0
  Fast50: 2.0

site10:
  Level1: 4.0

site11:
  Level1: 9.0

site12:
  Fast50: 2.0

site13:
  Level1: 5.0

site14:
  Level1: 7.0
  Fast50: 1.0

site15:
  Level1: 4.0
  Fast50: 1.0

site16:
  Level1: 3.0

site17:
  No chargers allocated at this site

site18:
  Level1: 2.0
  Fast50: 1.0

site19:
  Level1: 3.0
  Fast50: 2.0

site20:
  Level1: 3.0
  Fast50: 1.0

Objective Value: 438,000
Total Cost: 438,000
Total Vehicles Served: 940
Total Chargers Installed: 91



Note that we didn't have to do anything else other than re-run `setObjective`.

### Service Balance Between Standard and Fast Chargers
The vehicles served by fast chargers cannot exceed 60% of the vehicles served by standard chargers.
*Reminder: our objective function is still minimizing cost*

**Question**: Will this constraint change our current solution where we're maximizing the number of stations?

\begin{equation*}
\sum_{f \in F} v_f*x_f \le 0.6*\sum_{s \in S} v_s*x_s
\end{equation*}

In [406]:
### The vehicles served by fast chargers cannot exceed 60% of vehicles served by standard chargers
service_balance = model.addConstr(vehicles_all_fast <= 0.6 * vehicles_all_standard, name='service_balance')

solve_and_print_solution(model, x, chargers,sites)

Optimal solution found

------------------
Objective value: 454000.0

------------------
site1:
  Level1: 3.0

site2:
  Level1: 2.0
  Fast50: 2.0

site3:
  Level1: 8.0

site4:
  Level1: 18.0

site5:
  Level1: 2.0

site6:
  Level1: 15.0

site7:
  Level1: 13.0

site8:
  Level1: 10.0

site9:
  Level1: 4.0
  Fast50: 2.0

site10:
  Level1: 4.0

site11:
  Level1: 9.0

site12:
  Fast50: 2.0

site13:
  Level1: 5.0

site14:
  Level1: 17.0

site15:
  Level1: 14.0

site16:
  Level1: 3.0

site17:
  No chargers allocated at this site

site18:
  Level1: 12.0

site19:
  Level1: 3.0
  Fast50: 2.0

site20:
  Level1: 13.0

Objective Value: 454,000
Total Cost: 454,000
Total Vehicles Served: 940
Total Chargers Installed: 163



**Discussion**: The constraint didn't change anything! Why?
- Our current solution has 0 fast chargers (only Level 1s)
- The constraint is: $0 \le 0.6 \times (\text{positive number})$
- This is always satisfied - it's a **non-binding constraint**

**Key Learning**: Not all constraints are active in every solution! Some constraints only matter when certain decisions are made.

**To see this constraint in action**, we need a scenario where fast chargers might actually be installed. Let's switch back to maximizing vehicles served:

In [407]:
### Change objective back to maximize vehicles served
model.setObjective(vehicles_all, GRB.MAXIMIZE)
solve_and_print_solution(model, x, chargers,sites)

Optimal solution found

------------------
Objective value: 3818.0

------------------
site1:
  Level1: 40.0
  Level2: 1.0

site2:
  Level1: 36.0
  Level2: 1.0

site3:
  Level1: 54.0

site4:
  Level1: 29.0
  Level2: 4.0

site5:
  Level1: 44.0
  Level2: 1.0

site6:
  Level1: 58.0

site7:
  Level1: 29.0

site8:
  Level1: 61.0
  Level2: 1.0

site9:
  Level1: 40.0
  Level2: 1.0

site10:
  Level1: 43.0

site11:
  Level1: 55.0
  Level2: 1.0

site12:
  Level1: 35.0

site13:
  Level1: 50.0

site14:
  Level1: 30.0
  Level2: 1.0

site15:
  Level1: 65.0
  Level2: 1.0

site16:
  Level1: 48.0

site17:
  Level1: 45.0

site18:
  Level1: 58.0
  Level2: 1.0

site19:
  Level1: 38.0

site20:
  Level1: 64.0

Objective Value: 3,818
Total Cost: 1,922,000
Total Vehicles Served: 3,818
Total Chargers Installed: 935



**Now the constraint matters!** When maximizing vehicles served, we want fast chargers, but the service balance constraint limits how many we can install.

We stored each constraint as an object, which makes it easier to interact with these later. Now, let's remove the `service_balance` constraint.

In [408]:
model.remove(service_balance)

### Slack and Binding Constraints

The `Slack` of a constraint is the gap between the left-hand side and right-hand side values at the optimal solution. You can also think of this as how far the value is from its bound, in our case upper bound. Printing this for the `resource_usage` constraints will show how much of each resource is remaining.

In [409]:
for j in global_resources:
    print(f"Remaining {j}: {global_resource_constraints[j].Slack}")

Remaining budget: 0.0
Remaining equipment: 87.0


**Question**: Which one of the resource constraints is binding? Can a constraint be binding if the remaining resource value is not 0?

### Let's check in on our model and write it to an LP file
As you code a model, you may want to make sure adding variables, constraints, and the objective all go as planned.

In [410]:

### Writing a *.lp file is a great way to get a look at your model
model.write('our_model.lp')
model.write('our_model.mps')

### Meet minimum service requirements

Management has some operational requirements:
- We must have **at least 10 fast chargers** (any combination of Fast50, Fast150, or Ultra350) for customers who need quick charging
- We must have **at least 15 standard chargers** (any combination of Level1 or Level2) for customers with more time

Recall that $F = \{Fast50, Fast150, Ultra350\}$ and $S = \{Level1, Level2\}$.  Let \(T\) be the set of sites. We can model these minimum service requirements as:

\begin{align*}
\sum_{t \in T} \sum_{i \in F} x_{t,i} &\ge 10 \quad \text{(minimum fast chargers)} \\
\sum_{t \in T} \sum_{i \in S} x_{t,i} &\ge 15 \quad \text{(minimum standard chargers)}
\end{align*}

In [411]:

# At least 10 fast chargers across all sites
min_fast = model.addConstr(
    gp.quicksum(x[t, j] for t in sites for j in fast_chargers) >= 10,
    name="min_fast"
)

# At least 15 standard chargers across all sites
min_standard = model.addConstr(
    gp.quicksum(x[t, j] for t in sites for j in standard_chargers) >= 15,
    name="min_standard"
)
solve_and_print_solution(model, x, chargers, sites)

Optimal solution found

------------------
Objective value: 2800.0

------------------
site1:
  Level1: 3.0

site2:
  Level1: 27.0

site3:
  Level1: 50.0

site4:
  Level1: 18.0

site5:
  Level1: 45.0

site6:
  Level1: 58.0

site7:
  Fast50: 2.0

site8:
  Level1: 62.0

site9:
  Fast50: 3.0

site10:
  Level1: 4.0

site11:
  Level1: 14.0

site12:
  Fast50: 2.0

site13:
  Level1: 50.0

site14:
  Level1: 31.0

site15:
  Level1: 66.0

site16:
  Level1: 48.0

site17:
  No chargers allocated at this site

site18:
  Level1: 60.0

site19:
  Fast50: 3.0

site20:
  Level1: 64.0

Objective Value: 2,800
Total Cost: 1,380,000
Total Vehicles Served: 2,800
Total Chargers Installed: 610



### Cost of installing chargers:
- In the data frame above, we see there is an installation cost per charger. 
- Now management asks: **What's the cheapest way to serve at least 500 vehicles per day?**
- Write an expression for the total *variable cost* of chargers installed.
- Add a constraint that we must serve at least 500 vehicles per day.
- Make that the new objective to minimize total cost and solve.

Let $c_i$ be the installation cost of charger type $i$.
\begin{align*}
\text{total installation cost} = &\sum_i c_i*x_i \\
\text{Minimize} &\sum_i c_i*x_i \\
\text{subject to:} & \sum_i v_i*x_i \ge V_{min}
\end{align*}

Let $c_i$ be the installation cost of charger type $i$, and let T be the list of Sites
$$
\text{Minimize} \quad \sum_{t \in T} \sum_{i \in I} c_i \cdot x_{t,i}
$$

$$
\text{subject to:} \quad \sum_{t \in T} \sum_{i \in I} v_i \cdot x_{t,i} \ge 500
$$

In [412]:
### Variable installation cost
vCost_all_chargers = gp.quicksum(
    chargers_data.installation_cost[i] * x[t, i]
    for t in sites
    for i in chargers
)

### Add service level constraint - must serve at least 500 vehicles per day
min_service_level = 500

service_requirement = model.addConstr(
    vehicles_all >= min_service_level,
    name="service_requirement"
)

model.setObjective(vCost_all_chargers, GRB.MINIMIZE)

solve_and_print_solution(model, x, chargers, sites)


Optimal solution found

------------------
Objective value: 438000.0

------------------
site1:
  Level2: 1.0

site2:
  Level1: 2.0
  Fast50: 2.0

site3:
  Level1: 8.0

site4:
  Level1: 8.0
  Fast50: 1.0

site5:
  Level1: 2.0

site6:
  Level1: 5.0
  Fast50: 1.0

site7:
  Level1: 3.0
  Fast50: 1.0

site8:
  Fast50: 1.0

site9:
  Level1: 4.0
  Fast50: 2.0

site10:
  Level1: 4.0

site11:
  Level1: 9.0

site12:
  Fast50: 2.0

site13:
  Level1: 5.0

site14:
  Level1: 7.0
  Fast50: 1.0

site15:
  Level1: 4.0
  Fast50: 1.0

site16:
  Level2: 1.0

site17:
  No chargers allocated at this site

site18:
  Level1: 2.0
  Fast50: 1.0

site19:
  Level1: 3.0
  Fast50: 2.0

site20:
  Level1: 3.0
  Fast50: 1.0

Objective Value: 438,000
Total Cost: 438,000
Total Vehicles Served: 936
Total Chargers Installed: 87



### Fixed costs: More decision variables
Our installation crew can only install one type of charger at a time. When we want to switch to a different type, we incur permitting and setup costs, which adds a fixed cost to each charger type. Given this, we have to decide whether or not we want to install each type of charger.

**Key question**: Will adding fixed costs change which charger types we choose when trying to minimize cost while serving 500 vehicles/day?

Let's go step-by-step to add in the fixed costs for each charger type.
- First define a new set of variables indexed by charger type and call them `y`.
- Write a constraint that links the number of chargers installed with the new binary variable (**HINT:** use the cheat sheet).
- Don't resolve yet.

Let $f_i$ be the fixed cost of charger type $i$ and $y_i = 1$ if charger type $i$ is installed, $0$ otherwise.

If we decide to install a Fast50 charger, then $x_{Fast50} > 0$ and we want to have the associated cost go from 0 to the fixed cost.

So the constraint is
$$
x_{t,i} \le M_{t,i} \cdot y_{t,i} \quad \forall t \in T,\; i \in I
$$

But what do we choose for $M$?

In [413]:

print("=== Current Solution (without fixed costs) ===")

current_cost = vCost_all_chargers.getValue()
current_vehicles = vehicles_all.getValue()

print(f"Total cost: ${current_cost:,.0f}")
print(f"Vehicles served: {current_vehicles:.0f}")

print("\nCharger types installed by site:")
for s in sites:
    print(f"{s}:")
    for i in chargers:
        if x[s, i].X > 0.5:
            cost = chargers_data.installation_cost[i] * x[s, i].X
            print(f"  {i}: {x[s, i].X:.0f} units (cost: ${cost:,.0f})")
    print()

=== Current Solution (without fixed costs) ===
Total cost: $438,000
Vehicles served: 936

Charger types installed by site:
site1:
  Level2: 1 units (cost: $6,000)

site2:
  Level1: 2 units (cost: $4,000)
  Fast50: 2 units (cost: $36,000)

site3:
  Level1: 8 units (cost: $16,000)

site4:
  Level1: 8 units (cost: $16,000)
  Fast50: 1 units (cost: $18,000)

site5:
  Level1: 2 units (cost: $4,000)

site6:
  Level1: 5 units (cost: $10,000)
  Fast50: 1 units (cost: $18,000)

site7:
  Level1: 3 units (cost: $6,000)
  Fast50: 1 units (cost: $18,000)

site8:
  Fast50: 1 units (cost: $18,000)

site9:
  Level1: 4 units (cost: $8,000)
  Fast50: 2 units (cost: $36,000)

site10:
  Level1: 4 units (cost: $8,000)

site11:
  Level1: 9 units (cost: $18,000)

site12:
  Fast50: 2 units (cost: $36,000)

site13:
  Level1: 5 units (cost: $10,000)

site14:
  Level1: 7 units (cost: $14,000)
  Fast50: 1 units (cost: $18,000)

site15:
  Level1: 4 units (cost: $8,000)
  Fast50: 1 units (cost: $18,000)

site16:
  

In [414]:
y = model.addVars(sites, chargers, vtype=GRB.BINARY, name="install")

### Link binary variable to charger installation using `big-M` constraints
M = 100

# If y[t,i] = 0 → x[t,i] = 0
link_x_y_upper = model.addConstrs(
    (x[t, i] <= M * y[t, i] for t in sites for i in chargers),
    name="link_upper"
)

# If x[t,i] > 0 → y[t,i] = 1
link_x_y_lower = model.addConstrs(
    (x[t, i] >= y[t, i] for t in sites for i in chargers),
    name="link_lower"
)

model.update()

### Fixed costs: Combining costs
- Write an expression that finds the total fixed costs using the new variable and the `fixed_cost` column from `chargers_data`.
- Add that to the variable cost expression and set that as the new objective
- Solve!

**Question**: Do you think adding fixed costs will cause us to use fewer charger types?

\begin{align*}
&\text{total fixed cost} = \sum_i f_i*y_i \\
\text{total cost} = \space&\text{total fixed cost} + \text{total installation cost} = \sum_i f_i*y_i + c_i*x_i
\end{align*}

$$
\text{Total fixed cost} = \sum_{t \in T} \sum_{i \in I} f_i \cdot y_{t,i}
$$

$$
\text{Total cost} = \sum_{t \in T} \sum_{i \in I} \left( f_i \cdot y_{t,i} + c_i \cdot x_{t,i} \right)
$$

In [415]:
# Total fixed cost
fixed_cost_total = gp.quicksum(
    chargers_data.fixed_cost[i] * y[t, i]
    for t in sites
    for i in chargers
)

# Total variable cost (you already had this)
variable_cost_total = gp.quicksum(
    chargers_data.installation_cost[i] * x[t, i]
    for t in sites
    for i in chargers
)

# Combined objective
total_cost = fixed_cost_total + variable_cost_total

model.setObjective(total_cost, GRB.MINIMIZE)
solve_and_print_solution(model, x, chargers, sites)


Optimal solution found

------------------
Objective value: 655000.0

------------------
site1:
  Level1: 3.0

site2:
  Level1: 22.0

site3:
  Level1: 8.0

site4:
  Fast50: 2.0

site5:
  Level1: 2.0

site6:
  Level1: 15.0

site7:
  Level1: 13.0

site8:
  Level1: 10.0

site9:
  Fast50: 3.0

site10:
  Level1: 4.0

site11:
  Level1: 9.0

site12:
  Fast50: 2.0

site13:
  Level1: 5.0

site14:
  Level1: 17.0

site15:
  Level1: 14.0

site16:
  Level1: 3.0

site17:
  No chargers allocated at this site

site18:
  Level1: 12.0

site19:
  Fast50: 3.0

site20:
  Level1: 13.0

Objective Value: 655,000
Total Cost: 480,000
Total Vehicles Served: 1,000
Total Chargers Installed: 160



## Multi-objective optimization
- We are asked to maximize the total vehicles served while minimizing costs.
- Math optimization cannot **simultaneously** work on two objectives -- there's always a tradeoff.
- Two types of multi-objective: hierarchical, and weighted (blended).
- After talking more about how we want to prioritize the objectives, we want to:
    - First maximize the **vehicles served per day**.
    - Then minimize total costs while not decreasing vehicles served by more than 10%
- Also include the following constraints we already discussed:
    - Resource limits
    - Meet minimum requirements
    - Make the most Level2 chargers

### DIY hierarchical multi-objective optimization
- Sometimes, we have two objectives that we want to optimize for, gurobi allows you to do this quite easily!
- lets assume we want to maximize vehicles served (objective 1), but also want to minimize cost (objective 2)


In [416]:

# Objective 1 (highest priority): maximize vehicles served
model.setObjectiveN(
    vehicles_all,
    index=0,
    priority=2,
    name="maximize_vehicles"
)

# Objective 2 (lower priority): minimize cost
model.setObjectiveN(
    variable_cost_total,
    index=1,
    priority=1,
    name="minimize_cost"
)

# ================================
# SOLVE
# ================================

solve_and_print_solution(model, x, chargers, sites)

Optimal solution found

------------------
Objective value: 920.0

------------------
site1:
  Level2: 1.0

site2:
  Level1: 9.0
  Level2: 1.0
  Fast50: 1.0

site3:
  Level1: 5.0
  Level2: 1.0

site4:
  Level1: 5.0
  Level2: 1.0
  Fast50: 1.0

site5:
  Level1: 2.0

site6:
  Level1: 5.0
  Fast50: 1.0

site7:
  Level2: 1.0
  Fast50: 1.0

site8:
  Fast50: 1.0

site9:
  Level1: 4.0
  Fast50: 2.0

site10:
  Level1: 4.0

site11:
  Level1: 9.0

site12:
  Fast50: 2.0

site13:
  Level1: 5.0

site14:
  Level1: 4.0
  Level2: 1.0
  Fast50: 1.0

site15:
  Level1: 4.0
  Fast50: 1.0

site16:
  Level2: 1.0

site17:
  No chargers allocated at this site

site18:
  Level1: 9.0
  Level2: 1.0

site19:
  Level2: 1.0
  Fast50: 2.0

site20:
  Level2: 1.0
  Fast50: 1.0

Objective Value: 920
Total Cost: 442,000
Total Vehicles Served: 920
Total Chargers Installed: 89



### Logical constraints with binary variables
Now that we have binary decision variables that show which charger types are installed, we can model logical relationships between them. Model the following statements and write `gurobipy` code.

We won't solve a model with these.
- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).
- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).
- We install between 2 and 3 types of standard chargers.

- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).

$$
y_{t,\text{Fast150}} + y_{t,\text{Ultra350}} \ge 1 \quad \forall t \in T
$$

- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).

$$
y_{t,\text{Fast150}} + y_{t,\text{Ultra350}} = 1 \quad \forall t \in T
$$


- We install between 2 and 3 types of standard chargers.

$$
2 \le \sum_{i \in S} y_{t,i} \le 3 \quad \forall t \in T
$$

In [417]:
# We can install either Fast150 or Ultra350 chargers, and possibly both (at least one)
model.addConstrs(
    (y[t, 'Fast150'] + y[t, 'Ultra350'] >= 1 for t in sites),
    name="at_least_one_fast"
)

# We must install either Fast150 or Ultra350 chargers, but not both (exactly one)
model.addConstrs(
    (y[t, 'Fast150'] + y[t, 'Ultra350'] == 1 for t in sites),
    name="exactly_one_fast"
)

# We install between 2 and 3 types of standard chargers (per site)
model.addConstrs(
    (gp.quicksum(y[t, i] for i in standard_chargers) >= 2 for t in sites),
    name="min_standard_types"
)

model.addConstrs(
    (gp.quicksum(y[t, i] for i in standard_chargers) <= 3 for t in sites),
    name="max_standard_types"
)

{'site1': <gurobi.Constr *Awaiting Model Update*>,
 'site2': <gurobi.Constr *Awaiting Model Update*>,
 'site3': <gurobi.Constr *Awaiting Model Update*>,
 'site4': <gurobi.Constr *Awaiting Model Update*>,
 'site5': <gurobi.Constr *Awaiting Model Update*>,
 'site6': <gurobi.Constr *Awaiting Model Update*>,
 'site7': <gurobi.Constr *Awaiting Model Update*>,
 'site8': <gurobi.Constr *Awaiting Model Update*>,
 'site9': <gurobi.Constr *Awaiting Model Update*>,
 'site10': <gurobi.Constr *Awaiting Model Update*>,
 'site11': <gurobi.Constr *Awaiting Model Update*>,
 'site12': <gurobi.Constr *Awaiting Model Update*>,
 'site13': <gurobi.Constr *Awaiting Model Update*>,
 'site14': <gurobi.Constr *Awaiting Model Update*>,
 'site15': <gurobi.Constr *Awaiting Model Update*>,
 'site16': <gurobi.Constr *Awaiting Model Update*>,
 'site17': <gurobi.Constr *Awaiting Model Update*>,
 'site18': <gurobi.Constr *Awaiting Model Update*>,
 'site19': <gurobi.Constr *Awaiting Model Update*>,
 'site20': <gurobi.Co

### Conditional statements
- If we install Level1 chargers, then we **must** install Level2 chargers.
- If we install Level1 chargers, then we **must** install Level2 **and** Fast50 chargers.
- If we install Level1 chargers, then we **must** install Level2 **or** Fast50 chargers (or both).

- If we install Level1 chargers, then we must install Level2 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} \quad \forall t \in T
$$

- If we install Level1 chargers, then we must install Level2 and Fast50 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} \quad \forall t \in T
$$

$$
y_{t,\text{Level1}} \le y_{t,\text{Fast50}} \quad \forall t \in T
$$

$$
2 \cdot y_{t,\text{Level1}} \le y_{t,\text{Level2}} + y_{t,\text{Fast50}} \quad \forall t \in T
$$

- If we install Level1 chargers, then we must install Level2 or Fast50 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} + y_{t,\text{Fast50}} \quad \forall t \in T
$$

In [418]:
#### Logical constraint candidates:

### If we install Level1 chargers, then we must install Level2 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] for t in sites),
    name="level1_implies_level2"
)

### If we install Level1 chargers, then we must install Level2 and Fast50 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] for t in sites),
    name="level1_implies_level2"
)

model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Fast50'] for t in sites),
    name="level1_implies_fast50"
)

model.addConstrs(
    (2 * y[t, 'Level1'] <= y[t, 'Level2'] + y[t, 'Fast50'] for t in sites),
    name="level1_implies_both"
)

### If we install Level1 chargers, then we must install Level2 or Fast50 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] + y[t, 'Fast50'] for t in sites),
    name="level1_implies_either"
)

{'site1': <gurobi.Constr *Awaiting Model Update*>,
 'site2': <gurobi.Constr *Awaiting Model Update*>,
 'site3': <gurobi.Constr *Awaiting Model Update*>,
 'site4': <gurobi.Constr *Awaiting Model Update*>,
 'site5': <gurobi.Constr *Awaiting Model Update*>,
 'site6': <gurobi.Constr *Awaiting Model Update*>,
 'site7': <gurobi.Constr *Awaiting Model Update*>,
 'site8': <gurobi.Constr *Awaiting Model Update*>,
 'site9': <gurobi.Constr *Awaiting Model Update*>,
 'site10': <gurobi.Constr *Awaiting Model Update*>,
 'site11': <gurobi.Constr *Awaiting Model Update*>,
 'site12': <gurobi.Constr *Awaiting Model Update*>,
 'site13': <gurobi.Constr *Awaiting Model Update*>,
 'site14': <gurobi.Constr *Awaiting Model Update*>,
 'site15': <gurobi.Constr *Awaiting Model Update*>,
 'site16': <gurobi.Constr *Awaiting Model Update*>,
 'site17': <gurobi.Constr *Awaiting Model Update*>,
 'site18': <gurobi.Constr *Awaiting Model Update*>,
 'site19': <gurobi.Constr *Awaiting Model Update*>,
 'site20': <gurobi.Co

#### Dipose of the gurobipy environment - and you're done! 

In [ ]:
model.dispose()